# Existence graphs

This notebook only contains the user parameters and launches the plotting functions stored in `existence_graphs.py`.

In [2]:
import numpy as np
import pandas as pd

from existence_graphs import run_existence_graphs

## 1. User parameters

In [3]:
# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
df = pd.read_csv("updated_sample_2010_2015_correctionsampling.csv")

In [4]:
# ------------------------------------------------------------
# Dataframe information
# ------------------------------------------------------------
df_name = "posterior_birth_2010_2015"
df_name_for_title = "Posterior 2010 2015"

# Use "time_independent" if the dataframe contains birth/death/lifespan information
# and must be projected to several years.
# Use "time_dependent" if the dataframe already corresponds to one observed/projected year.
df_type = "time_independent"

df_years_to_project = [2010, 2015]
df_year_observed = None

output_folder = "birth_graphs"

# If the dataframe already contains weights, write its column name here.
# If None, all rows receive equal weight.
weight_column = None
weight_column_name = "weight"

# Graph parameters
bins_birth = 30
female_value = 1
male_value = 0

## 2. Mapping function

This function is only required when `df_type = "time_independent"`. It converts the original population dataframe into a time-dependent dataframe for one year `t`.

In [5]:
def mapping(df_time_indep, t):
    # Keep only Birth events
    df_birth = df_time_indep[df_time_indep["event_name"] == "Birth"]
    
    # Select relevant columns and rename for clarity
    df_birth = df_birth[["id", "main_start_date", "main_duration", "Sex"]].copy()
    df_birth.rename(
        columns={"main_start_date": "birth_date", "main_duration": "lifespan"},
        inplace=True,
    )
    
    # Map gender directly
    df_birth["gender"] = df_birth["Sex"]
    
    # Compute death time
    df_birth["death_time"] = df_birth["birth_date"] + df_birth["lifespan"]
    
    # Create alive status
    df_birth["alive_status"] = "Dead"
    df_birth["alive_status"] = np.where(
        df_birth["birth_date"] > t,
        "Not born",
        df_birth["alive_status"],
    )
    df_birth["alive_status"] = np.where(
        (df_birth["birth_date"] <= t) & (t < df_birth["death_time"]),
        "Alive",
        df_birth["alive_status"],
    )
    
    # Calculate age only for alive individuals
    df_birth["age"] = np.where(
        df_birth["alive_status"] == "Alive",
        t - df_birth["birth_date"],
        np.nan,
    )
    
    # Final time-dependent dataframe
    df_time_dep = df_birth[["id", "alive_status", "age", "gender"]].copy()
    
    return df_time_dep

## 3. Run all graphs

In [6]:
saved_files = run_existence_graphs(
    df=df,
    df_name=df_name,
    df_name_for_title=df_name_for_title,
    df_type=df_type,
    output_folder=output_folder,
    df_years_to_project=df_years_to_project,
    df_year_observed=df_year_observed,
    mapping=mapping,
    weight_column=weight_column,
    weight_column_name=weight_column_name,
    bins_birth=bins_birth,
    female_value=female_value,
    male_value=male_value,
    show=False,
)